# 10 — Locked evaluation

This notebook measures event detection, operational incident burden, delay and
localisation after the configuration is frozen. Development is the default.
Holdout can be opened only by setting both `EVALUATION_PARTITION=holdout` and
`OPEN_HOLDOUT=1`; the run then writes an immutable receipt tying the result to
the selected configuration.

No model, threshold, feature or consolidation rule is changed here.


## 1. Select the evaluation partition deliberately


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from datetime import datetime, timezone
import tempfile

import joblib
import pandas as pd
from IPython.display import display

from telco_anomaly.detectors import (
    alerts_from_score_file,
    fit_topology_reference,
    materialize_measurement_features,
    materialize_wide_partition,
    merge_score_files,
    partition_exposure,
    score_residual_file,
    score_topology_file,
)
from telco_anomaly.evaluation import evaluate_cases, form_cases
from telco_anomaly.io import (
    file_sha256,
    immutable_output_directory,
    load_config,
    read_json,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
PARTITION = os.getenv("EVALUATION_PARTITION", "development").lower()
OPEN_HOLDOUT = os.getenv("OPEN_HOLDOUT", "0") == "1"
if PARTITION not in {"development", "holdout"}:
    raise ValueError("EVALUATION_PARTITION must be development or holdout")
if PARTITION == "holdout" and not OPEN_HOLDOUT:
    raise PermissionError(
        "Holdout remains sealed. Set OPEN_HOLDOUT=1 only after selection is frozen."
    )

CORE_RUN_ID = os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v1")
FEATURE_RUN_ID = os.getenv("PON_FEATURE_RUN_ID", "synthetic_pon_features_v1")
MODEL_RUN_ID = os.getenv("PON_MODEL_RUN_ID", "synthetic_pon_models_v1")
TRUTH_RUN_ID = os.getenv("PON_TRUTH_RUN_ID", "synthetic_pon_truth_v1")
SELECTION_RUN_ID = os.getenv("PON_SELECTION_RUN_ID", "synthetic_pon_selection_v1")
INCIDENT_RUN_ID = os.getenv("PON_INCIDENT_RUN_ID", "synthetic_pon_incidents_v1")
EVALUATION_RUN_ID = os.getenv("PON_EVALUATION_RUN_ID", f"synthetic_pon_{PARTITION}_v1")

RUN_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
MODEL_ROOT = DATA_ROOT / "models" / "synthetic_pon" / MODEL_RUN_ID
TRUTH_ROOT = DATA_ROOT / "evaluation" / "synthetic_pon" / TRUTH_RUN_ID
SELECTION_ROOT = DATA_ROOT / "selection" / "synthetic_pon" / SELECTION_RUN_ID
INCIDENT_ROOT = DATA_ROOT / "incidents" / "synthetic_pon" / INCIDENT_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "results" / "synthetic_pon" / EVALUATION_RUN_ID

selected_path = SELECTION_ROOT / "selected_configuration.json"
if not selected_path.exists():
    raise RuntimeError("No selected configuration passed Notebook 07's development gates")
configuration = read_json(selected_path)
POLICY = load_config("alert_policy", project_root=PROJECT_ROOT)
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
topology_path = CORE_ROOT / "topology_memberships.parquet"
topology = pd.read_parquet(topology_path) if topology_path.exists() else pd.DataFrame()
truth_directory = TRUTH_ROOT / ("holdout_locked" if PARTITION == "holdout" else "development")

receipt = {
    "opened_at_utc": datetime.now(timezone.utc).isoformat(),
    "partition": PARTITION,
    "holdout_authorised": bool(PARTITION == "holdout" and OPEN_HOLDOUT),
    "selected_configuration_sha256": file_sha256(selected_path),
    "model_run_id": MODEL_RUN_ID,
    "selection_run_id": SELECTION_RUN_ID,
}
display(pd.Series(receipt, name="evaluation receipt").to_frame())


## 2. Reconstruct frozen incidents for the requested partition


In [ ]:
def alerts_and_cases(score_path):
    frames = [
        alerts_from_score_file(
            score_path,
            channel,
            configuration["thresholds"][channel],
            min_consecutive=configuration["persistence_observations"][channel],
            recovery_consecutive=configuration["recovery_observations"],
        )
        for channel in configuration["channels"]
    ]
    alerts = pd.concat(frames, ignore_index=True).sort_values("alert_start")
    alerts = alerts.reset_index(drop=True)
    alerts["alert_id"] = [f"A-{number:09d}" for number in range(1, len(alerts) + 1)]
    cases, members = form_cases(
        alerts,
        topology,
        gap_seconds=configuration["incident_quiet_period_seconds"],
        thresholds=configuration["thresholds"],
        shared_scope_models=("group_common_mode",),
    )
    return alerts, cases, members


model_manifest = read_json(MODEL_ROOT / "model_manifest.json")
if PARTITION == "development":
    score_path = MODEL_ROOT / model_manifest["score_files"]["development"]
    alerts, cases, members = alerts_and_cases(score_path)
else:
    # Holdout telemetry is transformed and scored with the frozen bundle only.
    bundle = joblib.load(MODEL_ROOT / "residual_bundle.joblib")
    feature_manifest = read_json(
        DATA_ROOT / "features" / "synthetic_pon" / FEATURE_RUN_ID / "feature_manifest.json"
    )
    with tempfile.TemporaryDirectory() as temporary_name:
        temporary = Path(temporary_name)
        wide = temporary / "holdout_wide.parquet"
        features = temporary / "holdout_features.parquet"
        self_scores = temporary / "holdout_self_scores.parquet"
        residuals = temporary / "holdout_residuals.parquet"
        score_path = temporary / "holdout_scores.parquet"
        split = materialize_wide_partition(
            CORE_ROOT, RUN_ROOT / "SPLITS", "holdout", catalogue, wide,
            lookback_seconds=feature_manifest["history_seconds"],
            memory_limit=os.getenv("DUCKDB_MEMORY_LIMIT", "2GB"),
            threads=int(os.getenv("DUCKDB_THREADS", "2")),
        )
        materialize_measurement_features(
            wide, catalogue, features,
            history_window_seconds=feature_manifest["history_seconds"],
            minimum_history_seconds=feature_manifest["minimum_history_seconds"],
            gap_tolerance=feature_manifest["gap_tolerance"],
            seasonal_periods=feature_manifest.get("seasonal_periods", {}),
            score_start=split["score_start"], score_end=split["score_end"],
        )
        score_residual_file(
            bundle, features, self_scores,
            cadence_seconds=model_manifest["cadence_seconds"],
            dispersion_window_seconds=model_manifest["dispersion_window_seconds"],
            cusum_allowance=POLICY["channels"]["persistent_drift"]["cusum_allowance"],
            residual_destination=residuals,
        )
        reference_path = MODEL_ROOT / "topology_reference.parquet"
        if model_manifest["topology_enabled"] and reference_path.exists():
            reference = pd.read_parquet(reference_path)
            topology_scores = temporary / "holdout_topology_scores.parquet"
            group_types = sorted(reference.loc[
                reference["channel"].eq("group_common_mode"), "group_type"
            ].unique())
            score_topology_file(
                residuals, topology, reference, topology_scores,
                peer_group_type=model_manifest["peer_level"],
                group_types=group_types,
                min_peers=load_config("topology", project_root=PROJECT_ROOT)["peer_policy"]["minimum_valid_peers"],
                min_group_entities=load_config("topology", project_root=PROJECT_ROOT)["group_policy"]["minimum_entities"],
                min_group_fraction=load_config("topology", project_root=PROJECT_ROOT)["group_policy"]["minimum_available_fraction"],
            )
            merge_score_files(self_scores, topology_scores, score_path)
        else:
            score_path = self_scores
        exposure = partition_exposure(score_path, "entity_day", model_manifest["cadence_seconds"])
        alerts, cases, members = alerts_and_cases(score_path)
if PARTITION == "development":
    exposure = partition_exposure(score_path, "entity_day", model_manifest["cadence_seconds"])


## 3. Open truth only after all decisions are frozen


In [ ]:
events = pd.read_parquet(truth_directory / "fault_events.parquet")
intervals = pd.read_parquet(truth_directory / "fault_entity_intervals.parquet")

result = evaluate_cases(
    cases,
    members,
    events,
    intervals,
    exposure_value=exposure,
    exposure_unit="entity_day",
    decision_horizon_seconds=POLICY["evaluation"]["default_decision_horizon_seconds"],
    topology_memberships=topology,
)

metrics = result["metrics"].copy()
metrics["display_metric"] = metrics["metric"].replace({
    "case_precision": "incident_precision",
    "false_cases_per_entity_day": "false_incidents_per_entity_day",
    "total_cases_per_entity_day": "total_incidents_per_entity_day",
})
primary = metrics.loc[metrics["display_metric"].isin(POLICY["evaluation"]["primary_metrics"])]
display(primary[["display_metric", "value", "ci_low", "ci_high", "numerator", "denominator"]])
display(result["fault_type_results"])


## 4. Publish the immutable result and limitations


In [ ]:
small_fault_types = result["fault_type_results"].loc[
    result["fault_type_results"]["reporting_status"].ne("estimable"), "fault_type"
].tolist()
evaluation_manifest = {
    **receipt,
    "configuration_status": configuration["status"],
    "scoreable_faults": events["fault_id"].nunique(),
    "exposure_entity_days": exposure,
    "incidents": len(cases),
    "small_fault_types_descriptive_only": small_fault_types,
    "model_changes_allowed": False,
    "limitations": [
        "Synthetic PON results validate injected mechanisms, not operator prevalence.",
        "Sparse fault-type rows are descriptive rather than stable estimates.",
        "Localisation is a topology-scope estimate, not causal root-cause proof.",
    ],
}

if OUTPUT_ROOT.exists():
    previous_receipt = read_json(OUTPUT_ROOT / "evaluation_receipt.json")
    if previous_receipt["selected_configuration_sha256"] != receipt["selected_configuration_sha256"]:
        raise ValueError("Existing evaluation used a different selected configuration")
    print("Using existing immutable evaluation:", OUTPUT_ROOT)
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        metrics.to_parquet(output / "metrics.parquet", index=False)
        result["fault_type_results"].to_parquet(output / "fault_type_results.parquet", index=False)
        result["domain_type_results"].to_parquet(output / "domain_type_results.parquet", index=False)
        result["localisation_results"].to_parquet(output / "localisation_results.parquet", index=False)
        result["fault_results"].to_parquet(output / "fault_results.parquet", index=False)
        write_json(output / "evaluation_receipt.json", receipt)
        write_json(output / "evaluation_manifest.json", evaluation_manifest)
    print("Saved locked evaluation:", OUTPUT_ROOT)
print("No modelling decision may be changed from this notebook's results.")
print("Next: 11_PUBLIC_DATASET_VALIDATION.ipynb")
